# RAG from Scratch to Production — Episode 9/20
## Query enhancement: fixing the question before searching

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/09_query_enhancement.ipynb)

Users don't write search queries. They write *"Does it roll over?"*, *"How often can I WFH?"*, *"My laptop got nicked"*. Before searching, the [production app](https://github.com/JakkulaMahendhar/rag-chatbot) uses an LLM to fix the question in two steps:

1. **Rewrite**: turn a follow-up into a standalone question ("it" → "the learning budget").
2. **Expand**: write 3 different search queries for it.

That's 2 LLM calls before the search even starts. On Episode 2, a reader asked whether both are really needed, and I promised to test cutting them down. This episode keeps that promise.

**In this episode:**
1. Measure what enhancement actually fixes, on 10 hard questions.
2. Run production's exact prompts, and count calls, tokens and time.
3. Try **merging** both steps into 1 call.
4. Try **skipping** the rewrite when there's no conversation yet.

**Needs an LLM:** Gemini (free key) or local Ollama, like Episodes 1–4.

## 0. Setup

Same `generate()` as earlier episodes, plus a counter for **calls, tokens and time**: that's what query enhancement costs.

In [ ]:
%pip install -q google-generativeai requests rank-bm25 sentence-transformers

In [1]:
import os, re, json, time, requests
import numpy as np

PROVIDER = os.getenv("LLM_PROVIDER", "gemini")   # "gemini" or "ollama"
GEMINI_MODEL = "gemini-3.6-flash"
OLLAMA_MODEL = "llama3.1"


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY") or __import__("getpass").getpass("GEMINI_API_KEY: ")


if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _model = genai.GenerativeModel(GEMINI_MODEL)

USAGE = []   # one (seconds, input tokens, output tokens) entry per LLM call


def generate(prompt: str) -> str:
    """One interface, two providers. Temperature 0 so results are repeatable."""
    start = time.perf_counter()
    if PROVIDER == "gemini":
        response = _model.generate_content(prompt, generation_config={"temperature": 0})
        text = response.text
        tokens_in = response.usage_metadata.prompt_token_count
        tokens_out = response.usage_metadata.candidates_token_count
    else:
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
                  "options": {"temperature": 0}},
            timeout=300,
        )
        r.raise_for_status()
        data = r.json()
        text, tokens_in, tokens_out = data["response"], data.get("prompt_eval_count", 0), data.get("eval_count", 0)
    USAGE.append((time.perf_counter() - start, tokens_in, tokens_out))
    return text.strip()


generate("Say OK")   # warm-up, so model loading doesn't count against the first test
USAGE.clear()
print(f"Using {PROVIDER}")

Using ollama


## 1. The search engine: Episode 8's hybrid

The same 8 handbook chunks, searched with Episode 8's best hybrid (both scores min-max scaled, 80% vector + 20% BM25).

When there are several search queries, each one contributes its **top chunk**, and the results are pooled. In production, the reranker (Episode 10) then picks the best from this pool. So the question for this episode is simple: **does the right chunk make it into the pool?**

In [2]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

CHUNKS = [
    "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week.",
    "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year. Book leave with form HR-12.",
    "Parental Leave\nNew parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected start date.",
    "Expenses\nThe meal allowance during business travel is capped at 40 USD per day. Travel expenses are claimed with form EXP-22 within 30 days. Claims above 500 USD need the finance director.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over. Requests are approved by your direct manager.",
    "Equipment\nEvery employee receives a ThinkPad X1 Carbon laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours.",
    "IT Support\nIf the VPN shows error E-4031, your access token has expired: reset it in the Okta portal. For error E-4032, contact the IT helpdesk.",
    "Security\nMulti-factor authentication is mandatory on all company accounts. Passwords must be at least 14 characters long and must never be reused.",
]

def title(chunk: str) -> str:
    return chunk.split("\n")[0]

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = model.encode(CHUNKS, normalize_embeddings=True)
bm25 = BM25Okapi([tokenize(c) for c in CHUNKS])

def minmax(x):
    return (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.zeros_like(x)

def hybrid(query: str):
    cosine = chunk_vectors @ model.encode(query, normalize_embeddings=True)
    keyword = np.array(bm25.get_scores(tokenize(query)))
    return 0.8 * minmax(cosine) + 0.2 * minmax(keyword)

def search(queries: list[str]) -> list[str]:
    """Top chunk for each query, pooled (no duplicates)."""
    pool = []
    for q in queries:
        best = title(CHUNKS[int(np.argmax(hybrid(q)))])
        if best not in pool:
            pool.append(best)
    return pool

## 2. Ten hard questions

Three kinds:
- **Follow-ups** that only make sense with the conversation before them.
- **Slang and abbreviations**: WFH, PTO, "got nicked", "OKs", "per diem".
- **Codes**, which already work. They're here to catch enhancement making things **worse**.

In [3]:
TESTS = [
    # (question, conversation so far, section that answers it)
    ("Does it roll over?",                     "user: How much is the learning budget?\nassistant: 1,200 USD per year.", "Learning Budget"),
    ("Which form do I need for that?",         "user: How long is parental leave?\nassistant: 16 weeks, fully paid.",   "Parental Leave"),
    ("And what about the other one, E-4032?",  "user: What does VPN error E-4031 mean?\nassistant: Your access token expired; reset it in Okta.", "IT Support"),
    ("How often can I WFH?",                   "", "Remote Work"),
    ("How many PTO days do I get a year?",     "", "Annual Leave"),
    ("My laptop got nicked, what now?",        "", "Equipment"),
    ("Who OKs my conference spending?",        "", "Learning Budget"),
    ("What's the per diem on business trips?", "", "Expenses"),
    ("Which form is HR-12?",                   "", "Annual Leave"),
    ("What is form EXP-22 for?",               "", "Expenses"),
]


def evaluate(name: str, prepare):
    """prepare(question, history) -> list of search queries. Prints one line per question."""
    USAGE.clear()
    hits = 0
    print(f"── {name}")
    for question, history, answer in TESTS:
        queries = prepare(question, history)
        hit = answer in search(queries)
        hits += hit
        print(f"   {'✅' if hit else '❌'} {question:<40} → {queries}")
    seconds = sum(u[0] for u in USAGE)
    tokens_in, tokens_out = sum(u[1] for u in USAGE), sum(u[2] for u in USAGE)
    print(f"   {hits}/10 found   {len(USAGE)} LLM calls   tokens in {tokens_in:,} / out {tokens_out:,}   {seconds:.1f}s")
    return {"found": hits, "calls": len(USAGE), "in": tokens_in, "out": tokens_out, "seconds": seconds}

## 3. Baseline: no enhancement

Search with the question exactly as the user typed it. Zero LLM calls.

In [4]:
results = {}
results["No enhancement"] = evaluate("No enhancement", lambda q, history: [q])

── No enhancement
   ✅ Does it roll over?                       → ['Does it roll over?']
   ❌ Which form do I need for that?           → ['Which form do I need for that?']
   ✅ And what about the other one, E-4032?    → ['And what about the other one, E-4032?']
   ❌ How often can I WFH?                     → ['How often can I WFH?']
   ✅ How many PTO days do I get a year?       → ['How many PTO days do I get a year?']
   ✅ My laptop got nicked, what now?          → ['My laptop got nicked, what now?']
   ✅ Who OKs my conference spending?          → ['Who OKs my conference spending?']
   ✅ What's the per diem on business trips?   → ["What's the per diem on business trips?"]
   ✅ Which form is HR-12?                     → ['Which form is HR-12?']
   ✅ What is form EXP-22 for?                 → ['What is form EXP-22 for?']
   8/10 found   0 LLM calls   tokens in 0 / out 0   0.0s


Two misses, for two different reasons:
- *"Which form do I need for **that**?"*: there's nothing to search for without the conversation.
- *"How often can I **WFH**?"*: Episode 5's problem. The embedding model barely knows the abbreviation.

## 4. Production: rewrite, then expand (2 LLM calls)

These are the production app's prompts (`app/services/query_enhancer.py` and `query_expander.py`), lightly trimmed. Keep the example array in the expand prompt: it's what shows the model the exact output format.

In [5]:
def strip_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()
    return text


def rewrite(question: str, history: str) -> str:
    return generate(f"""You are a query rewriting engine for a RAG system.
Rewrite the latest user question into a standalone search query.
Use previous conversation context if required.
Rules:
- Resolve pronouns like it, this, that, they, its.
- Add missing technical context.
- Preserve user intent.
- Do not answer the question.
- Return only the search query.

Conversation History:
{history or "No previous conversation."}

Current Question:
{question}

Standalone Search Query:""").strip().strip('"')


def expand(query: str, n: int = 3) -> list[str]:
    response = generate(f"""You are a search query generator.
Generate {n} search queries.
Return ONLY a JSON array.
Example:
[
  "Android activity lifecycle",
  "Android Activity component",
  "Activity lifecycle callbacks"
]
Rules:
- Return valid JSON only
- No markdown
- No explanation
- No code block
Original Query:
{query}""")
    try:
        queries = [q.strip() for q in json.loads(strip_fences(response)) if isinstance(q, str)]
    except Exception:
        queries = []
    queries = list(dict.fromkeys(q for q in queries if len(q) > 3))
    return queries[:n] or [query]


def production(question, history):
    return expand(rewrite(question, history))     # 2 calls, for every question


results["Production (2 calls)"] = evaluate("Production: rewrite + expand", production)

── Production: rewrite + expand


   ✅ Does it roll over?                       → ['budget rollover policy', 'learning budget rollover', 'rollover policy for learning budget']


   ✅ Which form do I need for that?           → ['Parental leave forms for 16 weeks', '16 week parental leave policy template', 'Parental leave application form 16 weeks']


   ✅ And what about the other one, E-4032?    → ['VPN error E-4032', 'E-4032 VPN error code', 'VPN error code E-4032 meaning']


   ✅ How often can I WFH?                     → ['WFH frequency', 'work from home schedule', 'remote work policy']


...

   ✅ My laptop got nicked, what now?          → ['laptop theft recovery process', 'laptop theft investigation tips', 'recover stolen laptop with serial number']


   ✅ Who OKs my conference spending?          → ['Who is responsible for approving conference expenses', 'Conference expense approval process', 'Who can approve company conference expenses']


   ✅ What's the per diem on business trips?   → ['per diem rates for business travel', 'business travel per diem rates by country', 'per diem reimbursement rates for business travel']


   ✅ Which form is HR-12?                     → ['HR-12 form requirements', 'HR-12 form submission process', 'HR-12 form template download']


   ✅ What is form EXP-22 for?                 → ['EXP-22 purpose', 'EXP-22 function', 'EXP-22 specification']
   10/10 found   20 LLM calls   tokens in 1,938 / out 392   10.6s


Both misses fixed: "that" became *parental leave*, and "WFH" became *remote work*. And the codes still work.

But it costs **2 LLM calls on every question**, before searching anything.

## 5. Idea 1: merge both steps into one call

One prompt that returns the standalone question **and** the search queries together, as JSON. The search pool uses all of them.

In [6]:
def merged(question, history, n=3):
    response = generate(f"""You prepare search queries for a company-documents search engine.

Conversation history:
{history or "No previous conversation."}

Latest user message:
{question}

Return ONLY a JSON object, no markdown:
{{"question": "<the latest message rewritten as a standalone question, resolving words like it/that from the history>",
  "searches": ["<{n} different search queries. Spell out abbreviations and slang. Keep codes, numbers and names exactly as written. Do not add topics the user did not mention>"]}}""")
    try:
        data = json.loads(strip_fences(response))
        return [data["question"]] + [s for s in data["searches"] if isinstance(s, str)][:n]
    except Exception:
        return [question]


results["Merged (1 call)"] = evaluate("Merged: one call", merged)

── Merged: one call


   ✅ Does it roll over?                       → ['Does the learning budget roll over from year to year?', 'learning budget rollover policy', 'annual learning budget carryover', 'budget rollover for employee training']


   ✅ Which form do I need for that?           → ['What form do I need for parental leave?', 'Parental leave form', 'Company documents for parental leave', 'Forms for 16 weeks parental leave']


   ✅ And what about the other one, E-4032?    → ['What does VPN error E-4032 mean?', 'VPN error E-4032 meaning', 'Okta access token expiration error codes', 'VPN error E-4032 resolution steps']


   ✅ How often can I WFH?                     → ['How often can I work from home?', 'How often can employees work from home', 'Company remote work policy frequency', 'Work from home schedule requirements']


...

   ✅ My laptop got nicked, what now?          → ['What should I do if my laptop was stolen?', 'laptop theft recovery process', 'what to do if laptop is stolen and reported to police', 'steps to take after laptop is stolen']


   ✅ Who OKs my conference spending?          → ['Who Approves My Conference Spending?', 'Who Approves Conference Spending', 'Person Responsible for Approving Conference Expenses', 'Conference Expense Approval Process']


   ✅ What's the per diem on business trips?   → ['What is the daily allowance for business trips?', 'per diem allowance for business trips', 'daily expense allowance for company travel', 'business trip daily allowance policy']


   ✅ Which form is HR-12?                     → ['What is the form HR Twelve?', 'form HR Twelve', 'HR Twelve form document', 'HR Twelve form template']


   ✅ What is form EXP-22 for?                 → ['What is form EXP-22 for?', 'Form EXP-22 purpose', 'What is the purpose of EXP-22 form', 'Information about EXP-22 document']
   10/10 found   10 LLM calls   tokens in 1,183 / out 522   11.3s


Half the calls. But compare the token counts with production: fewer tokens **in**, more tokens **out**. One call has to write a longer answer (a JSON object with 4 strings) than either of the two separate calls.

That matters, because:
- **Time** depends mostly on how much the model writes, not on how many calls it makes. So the merged call isn't faster.
- **Price:** on most paid APIs, output tokens cost several times more than input tokens.

Also read its queries above: one prompt doing two jobs is harder to control. Despite the instruction to keep codes exactly as written, models sometimes "helpfully" reword them. Look at *Which form is HR-12?* above: in my local run, `HR-12` became **`HR Twelve`** in every query. Embeddings still found the right section here, but exact-term search (BM25) can't match a code that's been rewritten.

## 6. Idea 2: only rewrite when there's something to rewrite

The rewrite step resolves "it" and "that" **from the conversation**. On the first message of a chat there is no conversation, so there's nothing to resolve. Skip it, and just expand:

In [7]:
def skip_rewrite_without_history(question, history):
    query = rewrite(question, history) if history else question   # 1 call on a first message, 2 on a follow-up
    return expand(query)


results["Skip rewrite (1–2 calls)"] = evaluate("Rewrite only when there's history", skip_rewrite_without_history)

── Rewrite only when there's history


   ✅ Does it roll over?                       → ['budget rollover policy', 'learning budget rollover', 'rollover policy for learning budget']


   ✅ Which form do I need for that?           → ['Parental leave forms for 16 weeks', '16 week parental leave policy template', 'Parental leave application form 16 weeks']


   ✅ And what about the other one, E-4032?    → ['VPN error E-4032', 'E-4032 VPN error code', 'VPN error code E-4032 meaning']


   ✅ How often can I WFH?                     → ['Remote work policies', 'Flexible work arrangements', 'Work from home benefits']


...

   ✅ My laptop got nicked, what now?          → ['what to do if laptop is stolen', 'laptop theft recovery process', 'report stolen laptop to police']


   ✅ Who OKs my conference spending?          → ['Conference spending approval process', 'Who approves conference expenses', 'Conference budget approval procedures']


   ✅ What's the per diem on business trips?   → ['per diem rates for business travel', 'business trip per diem calculator', 'per diem allowance for employees']


   ✅ Which form is HR-12?                     → ['HR-12 form', 'HR-12 application form', 'HR-12 form pdf']


   ✅ What is form EXP-22 for?                 → ['Form EXP-22 meaning', 'What is Form EXP-22 used for', 'Form EXP-22 purpose']
   10/10 found   13 LLM calls   tokens in 1,198 / out 309   7.8s


## 7. The scorecard

In [8]:
print(f"{'approach':<26} {'found':>6} {'calls':>6} {'tokens in':>10} {'tokens out':>11} {'time':>7}")
for name, r in results.items():
    print(f"{name:<26} {r['found']:>4}/10 {r['calls']:>6} {r['in']:>10,} {r['out']:>11,} {r['seconds']:>6.1f}s")

approach                    found  calls  tokens in  tokens out    time
No enhancement                8/10      0          0           0    0.0s
Production (2 calls)         10/10     20      1,938         392   10.6s
Merged (1 call)              10/10     10      1,183         522   11.3s
Skip rewrite (1–2 calls)     10/10     13      1,198         309    7.8s


In my local run (llama3.1): all three LLM approaches found 10/10. **Skipping the rewrite when there's no history** did it with about a third fewer calls, the fewest tokens, and the least time. Merging cut calls and input tokens but wrote the most output, and was the **slowest**. Your numbers with Gemini will differ, so check the table above.

**So, the promise:** yes, you can cut the calls, but not the way I first expected. The win isn't merging two prompts into one. It's **not calling the LLM when it has nothing to do**. In a real chat, most first messages have no history, so this saves a call on every new conversation.

## 8. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) (`app/services/rag_chat.py`):
- **Always** calls the rewrite step, even on the first message. Skipping it without history is a one-line change.
- Searches **only the expanded queries**: neither the user's original question nor the rewritten one is searched. Here that didn't hurt, but if expansion drifts (rewording a code, adding a topic), the user's own words are gone. Adding the original question to the search pool is cheap insurance.
- Strips markdown fences before parsing JSON, like the code above. Models wrap "JSON only" answers in fences anyway.

## Recap

| Step | What it fixes | Cost |
|---|---|---|
| Rewrite | Follow-ups ("that", "it") | 1 LLM call, **only useful with history** |
| Expand | Slang and abbreviations (WFH, PTO, per diem) | 1 LLM call |
| Merge both | Same fixes | 1 call, but longer output: not faster, harder to control |

**The rule:** the cheapest LLM call is the one you don't make. Skip steps that have nothing to do.

**Next, Episode 10: reranking.** Enhancement and hybrid search hand over a pool of candidates. A cross-encoder reads question and chunk **together** to pick the best.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)